# HexMagic : Overlays
# Terrain Overlay Component System

FastHTML-style composable terrain rendering inspired by FT component patterns.

## Overview

This design creates a declarative, composable system for rendering terrain with overlays, following FastHTML's pattern of combining components with positional and named parameters.

### Current State

Today, overlays are managed imperatively:

```python
def apply_overlays(zoomed: Terrain, result: ZoomResult, board: GameBoard,
                   c2f: dict, overlays: set, settle_attrs: dict = None):
    builder = zoomed.hexGrid.builder
    builder.adjust("borders", board.countries_overlay(zoomed, c2f))
    if 'cream' in overlays:
        zoomed.terrainCream()
    if 'elevation' in overlays:
        builder.adjust("elevation", zoomed.elevation_borders())
    # ... 10 more if statements
```

**Problems:**
- Imperative style requires checking membership in set
- Hard to reuse or compose in different contexts
- No clear hierarchy or ordering control
- HTMX attributes must be added separately
- Difficult to customize individual overlays

### Proposed State

```python
TerrainDisplay(
    terrain=demo_terrain,
    result=zoom_result,
    board=game_board,
    Cream(),
    Elevation(),
    Rivers(),
    ClimateDot(levels=4),
    Temperature(icon_size=8),
    Borders(windy=True),
    Names(font="Cinzel", size=14),
    Settlements(scale=2.0, attrs={'hx-get': '/settlement/{id}'}),
    hx_swap="outerHTML",
    id="terrain-map",
    cls="w-full h-full"
)
```

**Benefits:**
- Declarative — what you see is what renders
- Composable — mix and match overlays
- Orderable — layer order determined by position
- Customizable — each overlay can have its own params
- HTMX-ready — attributes flow through naturally

---

## Design

### Core Principle

**Components are functions that return overlay specifications**. The container renders them in order.

Following FastHTML patterns:
- **Positional params** = children (overlay components)
- **Named params** = attributes (terrain data, HTMX attrs)
- **Returns** = FastTag (Div containing HexTouchMap)

### Component Hierarchy

```
TerrainDisplay (Container)
    ├── Terrain context (terrain, result, board, c2f)
    ├── Overlay components (children)
    │   ├── Cream()
    │   ├── Elevation()
    │   ├── Rivers()
    │   ├── ClimateDot()
    │   ├── Temperature()
    │   ├── Watersheds()
    │   ├── Flow()
    │   ├── Borders()
    │   ├── Names()
    │   └── Settlements()
    └── Container attributes (id, cls, hx-*)
```

In [ ]:
#| default_exp overlay

In [ ]:
#| export
#| hide
#import nbdev; nbdev.nbdev_export()
import sys
import math
from fastcore.basics import patch

#| export
## Getting Started

In [ ]:
#| export
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex


from HexMagic.styles import StyleCSS,  SVGBuilder

In [ ]:
#| export
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.voronoi import generate_plate_terrain


Terrain.fromSeeds = generate_plate_terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo
from HexMagic.geology import Geology, DrainageBasins


In [ ]:
#| export
from HexMagic.terrainpatterns import TerrainPatterns
from HexMagic.climate import TerrainFactory

In [ ]:
#| export

from HexMagic.terraform import Terraform
from HexMagic.styles import apply_looping_animation, LoopingLayerAnimation


In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
import numpy as np
from dataclasses import dataclass

### Proposed State

```python
TerrainDisplay(
    terrain=demo_terrain,
    result=zoom_result,
    board=game_board,
    Cream(),
    Elevation(),
    Rivers(),
    ClimateDot(levels=4),
    Temperature(icon_size=8),
    Borders(windy=True),
    Names(font="Cinzel", size=14),
    Settlements(scale=2.0, attrs={'hx-get': '/settlement/{id}'}),
    hx_swap="outerHTML",
    id="terrain-map",
    cls="w-full h-full"
)
```

**Benefits:**
- Declarative — what you see is what renders
- Composable — mix and match overlays
- Orderable — layer order determined by position
- Customizable — each overlay can have its own params
- HTMX-ready — attributes flow through naturally

---

## Design

### Core Principle

**Components are functions that return overlay specifications**. The container renders them in order.

Following FastHTML patterns:
- **Positional params** = children (overlay components)
- **Named params** = attributes (terrain data, HTMX attrs)
- **Returns** = FastTag (Div containing HexTouchMap)

### Component Hierarchy

```
TerrainDisplay (Container)
    ├── Terrain context (terrain, result, board, c2f)
    ├── Overlay components (children)
    │   ├── Cream()
    │   ├── Elevation()
    │   ├── Rivers()
    │   ├── ClimateDot()
    │   ├── Temperature()
    │   ├── Watersheds()
    │   ├── Flow()
    │   ├── Borders()
    │   ├── Names()
    │   └── Settlements()
    └── Container attributes (id, cls, hx-*)
```

### TerrainDisplay Container

The main container that orchestrates rendering.

```python
def TerrainDisplay(
    *overlays,                    # Positional: overlay components
    terrain: Terrain = None,      # Required terrain data
    result: ZoomResult = None,    # Optional zoom result (for basins)
    board: GameBoard = None,      # Optional board (for kingdoms/settlements)
    c2f: dict = None,             # Optional coarse-to-fine mapper
    radius: int = None,           # Optional radius adjustment
    **attrs                       # All HTMX/HTML attributes
) -> FT:
    """
    Render terrain with composable overlay components.
    
    Usage:
        TerrainDisplay(
            terrain=t,
            Cream(),
            Rivers(),
            id="map",
            hx_swap="outerHTML"
        )
    """
    # Implementation creates HexTouchMap with all overlays applied
```

**Returns:** `Div(HexTouchMap(grid), **attrs)`

### Overlay Components

Each overlay is a function returning an overlay specification.

#### Base Pattern

```python
@dataclass
class OverlaySpec:
    """Specification for a single overlay layer."""
    name: str                              # Layer name for builder.adjust()
    renderer: Callable[[Context], str]     # Function that produces SVG
    requires: set[str] = field(default_factory=set)  # Dependencies
    priority: int = 50                     # Render order (lower = earlier)

class Context:
    """Rendering context passed to overlay renderers."""
    terrain: Terrain
    grid: HexGrid
    builder: SVGBuilder
    result: ZoomResult = None
    board: GameBoard = None
    c2f: dict = None
```

#### Example Overlay Components

```python
def Cream(**kw) -> OverlaySpec:
    """Terrain base coloring."""
    def render(ctx: Context) -> str:
        ctx.terrain.terrainCream()
        return ""  # Modifies grid in place
    return OverlaySpec("cream", render, priority=10)


def Elevation(show_labels: bool = True, **kw) -> OverlaySpec:
    """Elevation contour lines."""
    def render(ctx: Context) -> str:
        return ctx.terrain.elevation_borders(labels=show_labels)
    return OverlaySpec("elevation", render, priority=20)


def Rivers(**kw) -> OverlaySpec:
    """River network from watersheds."""
    def render(ctx: Context) -> str:
        if not ctx.result or not ctx.result.basins:
            return ""
        return ctx.result.basins.draw_watersheds()
    return OverlaySpec("rivers", render, requires={'result'}, priority=30)


def ClimateDot(levels: int = 3, **kw) -> OverlaySpec:
    """Climate zone dots."""
    def render(ctx: Context) -> str:
        return ctx.terrain.dottedClimate(levels=levels)
    return OverlaySpec("climate", render, priority=40)


def Temperature(icon_size: int = 8, **kw) -> OverlaySpec:
    """Temperature icon overlay."""
    def render(ctx: Context) -> str:
        return ctx.terrain.render_icon_temperature(size=icon_size)
    return OverlaySpec("temperature", render, priority=41)


def Watersheds(**kw) -> OverlaySpec:
    """Watershed boundary dots."""
    def render(ctx: Context) -> str:
        if not ctx.result or not ctx.result.basins:
            return ""
        return ctx.result.basins.dotted_watershed_overlay()
    return OverlaySpec("watersheds", render, requires={'result'}, priority=35)


def Flow(**kw) -> OverlaySpec:
    """Flow direction arrows."""
    def render(ctx: Context) -> str:
        return ctx.terrain.flow_diagram()
    return OverlaySpec("flow", render, priority=42)


def Borders(windy: bool = True, iterations: int = 2, **kw) -> OverlaySpec:
    """Kingdom borders."""
    def render(ctx: Context) -> str:
        if not ctx.board:
            return ""
        return ctx.board.countries_overlay(ctx.terrain, ctx.c2f)
    return OverlaySpec("borders", render, requires={'board'}, priority=60)


def Names(font: str = "Cinzel", size: int = 14, **kw) -> OverlaySpec:
    """Kingdom name labels."""
    def render(ctx: Context) -> str:
        if not ctx.board:
            return ""
        return ctx.board.names_overlay(ctx.terrain, ctx.c2f)
    return OverlaySpec("names", render, requires={'board'}, priority=70)


def Settlements(scale: float = 2.0, attrs: dict = None, **kw) -> OverlaySpec:
    """Settlement markers."""
    def render(ctx: Context) -> str:
        if not ctx.board:
            return ""
        overlay = ctx.board.settlementOverlay(ctx.terrain, ctx.c2f, scale=scale)
        if attrs:
            # Inject HTMX attributes into settlement pieces
            ctx.grid.builder.adjust("settlement_attrs", _inject_attrs(overlay, attrs))
        return overlay
    return OverlaySpec("settlements", render, requires={'board'}, priority=80)
```

---

## Implementation Strategy

### Phase 1: Core Components

Create the base system in a new module `HexMagic/display.py`:

```python
from dataclasses import dataclass, field
from typing import Callable, Any
from fasthtml.common import *

@dataclass
class Context:
    """Rendering context."""
    terrain: Terrain
    grid: HexGrid
    builder: SVGBuilder
    result: ZoomResult = None
    board: GameBoard = None
    c2f: dict = None

@dataclass
class OverlaySpec:
    """Overlay specification."""
    name: str
    renderer: Callable[[Context], str]
    requires: set[str] = field(default_factory=set)
    priority: int = 50

def TerrainDisplay(*overlays, terrain, result=None, board=None, 
                   c2f=None, radius=None, **attrs):
    """Main container."""
    # 1. Prepare context
    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)
    
    ctx = Context(
        terrain=terrain,
        grid=grid,
        builder=grid.builder,
        result=result,
        board=board,
        c2f=c2f
    )
    
    # 2. Validate requirements
    available = {'terrain', 'grid', 'builder'}
    if result: available.add('result')
    if board: available.add('board')
    if c2f: available.add('c2f')
    
    # 3. Sort by priority and render
    specs = sorted(overlays, key=lambda o: o.priority)
    for spec in specs:
        missing = spec.requires - available
        if missing:
            logging.warning(f"Skipping {spec.name}: missing {missing}")
            continue
        
        try:
            svg = spec.renderer(ctx)
            if svg:
                grid.builder.adjust(spec.name, svg)
        except Exception as e:
            logging.error(f"Overlay {spec.name} failed: {e}")
    
    # 4. Update and return
    grid.update()
    return Div(
        HexTouchMap(grid, cls=attrs.pop('map_cls', 'w-full h-full')),
        **attrs
    )
```

### Phase 2: Overlay Library

Add all overlay components to `HexMagic/display.py`:

```python
# Terrain base
def Cream(**kw): ...
def Elevation(**kw): ...

# Water
def Rivers(**kw): ...
def Watersheds(**kw): ...
def Flow(**kw): ...

# Climate
def ClimateDot(**kw): ...
def Temperature(**kw): ...

# Political
def Borders(**kw): ...
def Names(**kw): ...
def Settlements(**kw): ...

# Pieces
def PiecePlan(piece, num_rounds=10, color="#4CAF50", **kw): ...
```

### Phase 3: Route Integration

Update routes to use the new system:

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str, rings: int = None):
    uid = ensure_user(session)
    active = globalStore.active_board(uid)
    if not active:
        return P("No active game", id="map")
    
    settlement = globalStore.settlement_from_id(id)
    region = settlement.region(active.board.terrain.hexGrid, rings=rings or 5)
    result = active.cover.zoom_region_fast(region, compute_weather=True)
    
    # Old way:
    # apply_overlays(result.terrain, result, active.board, result.c2f, overlays, attrs)
    
    # New way:
    overlays = get_overlays(session)
    components = []
    if 'cream' in overlays: components.append(Cream())
    if 'elevation' in overlays: components.append(Elevation())
    if 'rivers' in overlays: components.append(Rivers())
    if 'climate' in overlays: components.append(ClimateDot(levels=4))
    if 'temperature' in overlays: components.append(Temperature())
    # ... etc
    
    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )
```

### Phase 4: Preset Compositions

Create common combinations:

```python
def TerrainBase(terrain, **kw):
    """Basic terrain display."""
    return TerrainDisplay(
        Cream(),
        Elevation(),
        terrain=terrain,
        **kw
    )

def WorldMap(terrain, board, **kw):
    """Full world map with kingdoms."""
    return TerrainDisplay(
        Cream(),
        Elevation(),
        Borders(),
        Names(),
        Settlements(),
        terrain=terrain,
        board=board,
        **kw
    )

def DetailedRegion(terrain, result, board, **kw):
    """Region view with weather and rivers."""
    return TerrainDisplay(
        Cream(),
        Elevation(),
        Rivers(),
        ClimateDot(levels=4),
        Temperature(),
        Borders(),
        Settlements(attrs={'hx-get': '/settlement/{id}'}),
        terrain=terrain,
        result=result,
        board=board,
        c2f=result.c2f,
        **kw
    )
```

---

## Advanced Features

### Custom Overlays

Users can create their own overlay components:

```python
def MyCustomOverlay(color="#FF0000", **kw):
    """Draw circles at high elevations."""
    def render(ctx: Context) -> str:
        svg = ""
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > 100:
                center = ctx.grid.hexes[i].center
                svg += f'<circle cx="{center.x}" cy="{center.y}" r="5" fill="{color}"/>\n'
        return svg
    return OverlaySpec("custom", render, priority=90)

# Use it:
TerrainDisplay(
    terrain=t,
    Cream(),
    MyCustomOverlay(color="#00FF00")
)
```

### Conditional Overlays

```python
def ConditionalDisplay(show_weather=False, show_rivers=True, **kw):
    """Display with conditional overlays."""
    components = [Cream(), Elevation()]
    if show_weather:
        components.extend([ClimateDot(), Temperature()])
    if show_rivers:
        components.append(Rivers())
    
    return TerrainDisplay(*components, **kw)
```

### Overlay Groups

```python
class WeatherGroup:
    """Group of related overlays."""
    @staticmethod
    def components(levels=3, icon_size=8):
        return [
            ClimateDot(levels=levels),
            Temperature(icon_size=icon_size),
        ]

# Use it:
TerrainDisplay(
    terrain=t,
    Cream(),
    *WeatherGroup.components(levels=4),
    Rivers()
)
```

### Interactive Overlay Toggle

```python
@rt("/toggle_overlay/{id}")
def toggle_overlay(session, id: str, overlay: str):
    current = get_overlays(session)
    current ^= {overlay}  # Toggle
    set_overlays(session, current)
    
    # Build component list from session state
    components = overlay_set_to_components(current)
    
    active = globalStore.active_board(ensure_user(session))
    settlement = globalStore.settlement_from_id(id)
    region = settlement.region(active.board.terrain.hexGrid, rings=5)
    result = active.cover.zoom_region_fast(region)
    
    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map"
    )

def overlay_set_to_components(overlays: set) -> list[OverlaySpec]:
    """Convert session overlay set to component list."""
    mapping = {
        'cream': Cream(),
        'elevation': Elevation(),
        'rivers': Rivers(),
        'climate': ClimateDot(levels=4),
        'temperature': Temperature(),
        'watersheds': Watersheds(),
        'flow': Flow(),
        'borders': Borders(),
        'names': Names(),
        'settlements': Settlements(attrs={'hx-get': '/settlement/{id}'}),
    }
    return [mapping[k] for k in overlays if k in mapping]
```

---

## Migration Path

### Stage 1: Parallel Implementation

Keep `apply_overlays()` working while building the new system:

```python
# Old route (unchanged)
@rt("/old_map/{id}")
def old_settlement_map(session, id: str):
    # Uses apply_overlays()
    pass

# New route (using TerrainDisplay)
@rt("/new_map/{id}")
def new_settlement_map(session, id: str):
    # Uses TerrainDisplay()
    pass
```

### Stage 2: Gradual Conversion

Convert routes one at a time:
1. `/settlement_map/{id}` → TerrainDisplay
2. `/kingdom_map/{id}` → TerrainDisplay
3. `/world_map` → TerrainDisplay
4. `/piece_map/{id}` → TerrainDisplay

### Stage 3: Deprecation

Once all routes are converted:
1. Mark `apply_overlays()` as deprecated
2. Add migration guide to docs
3. Remove old code in next major version

---

## Testing Strategy

### Unit Tests

```python
def test_cream_overlay():
    terrain = create_test_terrain()
    spec = Cream()
    assert spec.name == "cream"
    assert spec.priority == 10

def test_terrain_display_basic():
    terrain = create_test_terrain()
    result = TerrainDisplay(
        Cream(),
        terrain=terrain,
        id="test"
    )
    assert isinstance(result, Div)
    # Check that terrain was rendered with cream

def test_missing_requirements():
    terrain = create_test_terrain()
    # Rivers requires result, which is None
    result = TerrainDisplay(
        Rivers(),
        terrain=terrain
    )
    # Should skip Rivers without crashing
```

### Integration Tests

```python
def test_full_settlement_map():
    active = create_test_game()
    settlement = active.board.kingdoms[0].settlements[0]
    region = settlement.region(active.board.terrain.hexGrid, rings=5)
    result = active.cover.zoom_region_fast(region)
    
    display = TerrainDisplay(
        Cream(),
        Elevation(),
        Rivers(),
        Borders(),
        Names(),
        Settlements(),
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map"
    )
    
    # Verify all layers are present
    svg = display.children[0].grid.builder.xml()
    assert "cream" in svg
    assert "rivers" in svg
```

---

## API Reference

### TerrainDisplay

```python
TerrainDisplay(
    *overlays: OverlaySpec,
    terrain: Terrain,
    result: ZoomResult = None,
    board: GameBoard = None,
    c2f: dict = None,
    radius: int = None,
    **attrs
) -> Div
```

**Parameters:**
- `*overlays` - Overlay components to render
- `terrain` - Required Terrain object
- `result` - Optional ZoomResult (needed for Rivers, Watersheds)
- `board` - Optional GameBoard (needed for Borders, Names, Settlements)
- `c2f` - Optional coarse-to-fine mapper for zoomed views
- `radius` - Optional radius adjustment
- `**attrs` - HTML/HTMX attributes for container Div

**Returns:** `Div` containing `HexTouchMap`

### Overlay Components

All overlays follow the same signature:

```python
OverlayName(**params) -> OverlaySpec
```

| Component | Parameters | Requires | Priority |
|-----------|------------|----------|----------|
| `Cream()` | - | terrain | 10 |
| `Elevation(show_labels=True)` | show_labels | terrain | 20 |
| `Rivers()` | - | result | 30 |
| `Watersheds()` | - | result | 35 |
| `ClimateDot(levels=3)` | levels | terrain | 40 |
| `Temperature(icon_size=8)` | icon_size | terrain | 41 |
| `Flow()` | - | terrain | 42 |
| `Borders(windy=True)` | windy, iterations | board | 60 |
| `Names(font="Cinzel", size=14)` | font, size | board | 70 |
| `Settlements(scale=2.0, attrs=None)` | scale, attrs | board | 80 |

---

## Examples

### Basic Terrain

```python
TerrainDisplay(
    Cream(),
    Elevation(),
    terrain=demo_terrain,
    id="basic-map"
)
```

### Settlement Detail View

```python
TerrainDisplay(
    Cream(),
    Elevation(),
    Rivers(),
    ClimateDot(levels=4),
    Borders(),
    Settlements(
        scale=2.0,
        attrs={'hx-get': f'/settlement_detail/{settlement.id}'}
    ),
    terrain=zoomed_terrain,
    result=zoom_result,
    board=game_board,
    c2f=zoom_result.c2f,
    id="settlement-map",
    hx_swap="outerHTML"
)
```

### Piece Planning View

```python
# Simulate piece movement
steps, region = piece.plan_region(grid, terrain.elevations, num_rounds=20)
result = cover.zoom_region_fast(region)

TerrainDisplay(
    Cream(),
    Elevation(),
    Rivers(),
    PiecePlan(piece=piece, steps=steps, color="#e74c3c"),
    Borders(),
    terrain=result.terrain,
    result=result,
    board=board,
    c2f=result.c2f,
    id="piece-map"
)
```

### Custom Overlay

```python
def HighElevationMarkers(threshold=100, color="#FF0000"):
    def render(ctx):
        svg = ""
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                svg += f'<circle cx="{c.x}" cy="{c.y}" r="3" fill="{color}"/>\n'
        return svg
    return OverlaySpec("high_elev", render, priority=90)

TerrainDisplay(
    Cream(),
    Elevation(),
    HighElevationMarkers(threshold=150, color="#00FF00"),
    terrain=demo_terrain
)
```

---

## Open Questions

1. **Performance**: Should we cache rendered overlays if context hasn't changed?
2. **Ordering**: Should priority be explicit or derived from list order?
3. **Error handling**: Silent skip vs. visible error indicator?
4. **Preset system**: Should presets be classes or functions?
5. **HTMX integration**: How to handle settlement click handlers elegantly?

## Future Enhancements

- **Overlay animations**: Fade in/out when toggling
- **Conditional rendering**: Show/hide based on zoom level
- **Layer groups**: Bundle related overlays (weather, political, etc.)
- **Overlay inspector**: Debug view showing render order and dependencies
- **Performance profiling**: Track render time per overlay

That's a much cleaner contract. So the rules are:

- **Allowed**: mutate `terrain` / `grid` hex styles (e.g. `colorMap()`, `terrainCream()`)
- **Allowed**: return an SVG string
- **Not allowed**: touch `builder` at all inside a renderer

So `TerrainDisplay` becomes the sole owner of the builder, and the flow is:

1. Each overlay renderer runs → side effects on hex styles + returns SVG str (or `""`)
2. `TerrainDisplay` calls `grid.update()` once to bake all hex style mutations
3. `TerrainDisplay` calls `builder.adjust(name, svg)` for any non-empty SVG strings

That means we need to audit the existing terrain methods. For example, if `elevation_borders()` currently calls `builder.adjust()` internally, we'd need a version that just returns the SVG string without touching the builder.

This leads to a natural split:
- **Compute methods** on `Terrain` that return SVG strings (what overlays call)
- **Builder methods** that compose and adjust layers (only `TerrainDisplay` touches)

Want to start by checking what `terrainCream()` and `colorMap()` currently do under the hood, so we know what needs reworking? If you can paste or run `inspect.getsource(terrain.terrainCream)` we can see if there are any builder side effects to clean up.

That's a clean rule. So the contract becomes:

- **Allowed**: mutate hex styles on `terrain` / `grid`
- **Allowed**: return an SVG string
- **Allowed**: `builder.add_definition()`, `builder.add_style()`, `builder.add_font()`
- **Not allowed**: `builder.adjust()` — only `TerrainDisplay` calls that

So `dottedClimate` is actually already ✅ — it only calls `add_definition()` which is permitted.

And `terrainCream` just needs the one `builder.adjust()` call removed from the end. The rest (the `add_style()` calls and hex style mutations) are all fine. We'd need a slim wrapper:

```python
def terrainCreamBase(self: Terrain):
    """Like terrainCream but without the builder.adjust() — for use with TerrainDisplay."""
    # ... same as terrainCream but drop the last two lines:
    # overlay = grid.styleLayerOrdered(...)
    # self.builder.adjust(layer_name, overlay)
```

Or alternatively, just have the `Cream` overlay call `terrainCream` but then remove the layer it injected. The cleaner fix is a new method. Want to patch `terrainCream` in this notebook to strip the `builder.adjust` call out?

In [ ]:
!cat ../docs/ll*

In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

Can you start to build TerrainDisplay

In [ ]:
#| export
#| export
import logging
from dataclasses import dataclass, field
from typing import Callable, Optional

@dataclass
class OverlayContext:
    """Rendering context passed to overlay renderers."""
    terrain: 'Terrain'
    grid: 'HexGrid'
    builder: 'SVGBuilder'
    result: 'ZoomResult' = None
    board: 'GameBoard' = None
    c2f: dict = None

@dataclass
class OverlaySpec:
    """Specification for a single overlay layer."""
    name: str
    renderer: Callable[['OverlayContext'], str]
    requires: set = field(default_factory=set)
    priority: int = 50

def TerrainDisplay(
    *overlays: OverlaySpec,
    terrain: 'Terrain',
    result=None,
    board=None,
    c2f: dict = None,
    radius: int = None,
    **attrs
):
    """Render terrain with composable overlay components."""
    grid = terrain.hexGrid
    if radius: grid.adjustRadius(radius)

    ctx = OverlayContext(
        terrain=terrain, grid=grid, builder=grid.builder,
        result=result, board=board, c2f=c2f
    )

    # Check what's available for dependency resolution
    available = {'terrain', 'grid', 'builder'}
    if result: available.add('result')
    if board:  available.add('board')
    if c2f:    available.add('c2f')

    # Sort by priority and render each overlay
    for spec in sorted(overlays, key=lambda o: o.priority):
        missing = spec.requires - available
        if missing:
            logging.warning(f"Skipping {spec.name}: missing {missing}")
            continue
        try:
            svg = spec.renderer(ctx)
            if svg: grid.builder.adjust(spec.name, svg)
        except Exception as e:
            logging.error(f"Overlay {spec.name} failed: {e}")

    grid.update()
    return Div(
        HexTouchMap(grid, cls=attrs.pop('map_cls', 'w-full h-full')),
        **attrs
    )


Lets build cream and elevation

Ah for elevations it is Terrain.colorMap and terrain.hexGrid.update

HexGrid.update??

So I think we are going to want to do something similar where we use the underlying style hexes

I think we want the overlay system to take in a set of parameters and generate svgstr. so the acceptable side effects are modifying the terrain and hexgrid (like with color map and creme), but we don't want to use methods that add layers to builder as part of the process. that is what the OverlayContext does. It might mean we need to rework some of these systems (like cream), but we are going to have a much more robust system in the end if there aren't side effects in the builder

In [ ]:
Terrain.colorMap??

In [ ]:
Terrain.elevation_borders??

More?

In [ ]:
Terrain.terrainCream??


In [ ]:
Terrain.dottedClimate??

In [ ]:
DrainageBasins.draw_watersheds??

In general we are ok, but we do need to check

We are going to allow adding definitiions, fonts, and styles to the builder. just not layers